# 00 — Hello, DSPy: programs, not prompts

**What you'll learn**

- The one idea behind everything in DSPy: an LLM pipeline is a *program* with learnable prompts, and you improve it the way you improve a neural network — with data, a loss, and an optimizer.
- How to connect a language model (`dspy.LM`, `dspy.configure`, `dspy.context`).
- Why reruns of these notebooks are almost free (caching).
- The PyTorch ↔ DSPy dictionary we will use for the entire course.

*Time: ~15 min. Cost: < $0.01 at the default model. Cached reruns are free.*

## Why DSPy exists

If you have built anything with LLMs, you have lived this loop: write a prompt, eyeball the output, tweak an adjective, break a different case, repeat. The prompt becomes a load-bearing string literal that nobody dares touch. Change the model and you start over.

DSPy's claim is that this is a compiler problem. You should write down **what** each LLM call takes in and produces (a typed signature), compose calls with ordinary Python into a **program**, and then let an **optimizer** search for the prompts — the wording, the few-shot examples — that maximize a **metric** you define on **data** you provide. Hand-prompting is assembly; DSPy wants you writing in a high-level language and compiling.

That is the whole course in one sentence. Here is the road we'll take:

| # | Notebook | You learn |
|---|----------|-----------|
| 00 | Hello, DSPy | the big idea, LM setup, caching |
| 01 | Signatures & Predict | typed I/O contracts for LM calls |
| 02 | Under the hood | how signatures become prompts; what optimizers tune |
| 03 | Modules & the graph | compose LLM + plain Python into a program |
| 04 | Data & evaluation | Examples, metrics (the "loss"), `Evaluate` |
| 05 | Bootstrap few-shot | the program that teaches itself demos |
| 06 | Instruction optimization | COPRO and MIPROv2 rewrite the words |
| 07 | Feedback & GEPA | textual feedback as a gradient |
| 08 | Agents & inference-time | ReAct tools, Refine/BestOfN |
| 09 | The optimizer landscape | every optimizer, and when to use which |
| 10 | Capstone | the full workflow on real data (HotPotQA) |

## The dictionary

You already know how to train models. DSPy reuses that entire mental model — only the parameters change from floating-point weights to *pieces of prompts*:

| PyTorch | DSPy | |
|---|---|---|
| `nn.Module` / `forward()` | `dspy.Module` / `forward()` | your program; a define-by-run graph |
| layer shapes `(in, out)` | signature `"question -> answer"` | the typed interface of one LM call |
| weights θ | **instructions + few-shot demos** per predictor | what gets learned |
| loss function | `metric(example, prediction)` | you define what "good" means |
| `optimizer.step()` | `teleprompter.compile(program, trainset=...)` | search over prompt space |
| eval loop | `dspy.Evaluate(devset=..., metric=...)` | parallel scoring harness |
| `state_dict()` | `program.save("prog.json")` | learned parameters, human-readable |

The training loop is the same loop:

```
PyTorch:  data -> model(x)   -> outputs    -> loss   -> backprop  -> updated weights
DSPy:     data -> program(x) -> Prediction -> metric -> optimizer -> updated prompts
```

One honest difference to keep in mind: prompts are discrete, so there is no gradient. Every DSPy optimizer is some flavor of *search* — generate candidate prompts, measure them with your metric, keep the best. Notebooks 05–07 are a tour of increasingly clever search strategies.

## Setup

From the repo root, in a fresh virtual environment:

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env        # then edit .env: your API key, and (optionally) your models
jupyter lab
```

These notebooks default to [OpenRouter](https://openrouter.ai) because one API key gives you every major model, which makes the "swap the model, keep the program" lesson cheap to try. Create a key at openrouter.ai → Keys and paste it into `.env`.

The config cell below opens every notebook in this course, and its first move is reading `.env` — which makes your model choice **global**: set `MODEL` (and `STRONG_MODEL`, the pricier model a few notebooks use as a teacher or reflection LM) once, and all twelve notebooks follow. No `.env`? The cell falls back to sensible defaults and prompts for the key. Want a different model in *one* notebook only? Uncomment an override line there — code beats `.env`.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

### Phoenix observability (optional)

Run the next cell to stream this notebook's LM calls into a local [Phoenix](https://phoenix.arize.com) trace UI — or skip it; nothing below depends on tracing. It finds or starts the Phoenix server, prints the link, and names the session by date (`dspy-lab-YYYY-MM-DD`) so different days never mix. Notebook 11 tells the full story.

In [ ]:
# optional: Phoenix tracing (see notebook 11)
import obs

obs.enable_phoenix()

### MLflow observability (optional)

Run the next cell to record this notebook's LM calls with MLflow instead (or as well): logs land in a repo-local store that is safe against remnants and reruns, browsable any time with `obs.mlflow_ui()`. Equally optional; equally explained in notebook 11.

In [ ]:
# optional: MLflow tracing (see notebook 11)
import obs

obs.enable_mlflow()

## The bottom layer: a raw LM call

`dspy.LM` wraps any model behind one interface (routing is delegated to LiteLLM — the `"provider/model"` string decides where the call goes). You can always call it directly, like a plain client:

In [ ]:
lm("In one short sentence, what does a lighthouse keeper do?")

> **What you should see:** a `list[str]` with one completion, e.g. `['A lighthouse keeper maintains the light and warns ships away from the coast.']` (wording will vary).
>
> This is the floor of the abstraction stack: no prompt engineering, no parsing, just text in, text out. Everything DSPy adds — typed fields, composition, optimization — is layered on top of this call. When something confuses you in later notebooks, remember it all bottoms out here.

In [ ]:
lm("In one short sentence, what does a lighthouse keeper do?")   # run it again

> **What you should see:** the *identical* string, instantly.
>
> That is the disk cache. `dspy.LM(..., cache=True)` is the default: every (model, prompt, settings) request is cached in `~/.dspy_cache`, so re-running a cell — or a whole notebook — costs nothing and returns the same result. This matters for this course: you can interrupt any notebook and rerun it from the top for free. `dspy.configure_cache(...)` adjusts limits and location; `cache=False` opts a model out.

## Global vs. scoped configuration

`dspy.configure(lm=...)` sets the default model for everything that follows. Sometimes you want a different model *for one block* — a stronger model for a hard step, a cheap one for a bulk step. That is `dspy.context`:

In [ ]:
with dspy.context(lm=dspy.LM(STRONG_MODEL, max_tokens=1000)):
    strong_answer = dspy.Predict("question -> answer")(question="What is a spring tide?")

cheap_answer = dspy.Predict("question -> answer")(question="What is a spring tide?")
strong_answer, cheap_answer

> **What you should see:** two `Prediction(answer=...)` objects — the first produced by the strong model, the second by the default one. (`dspy.Predict` is fully explained in notebook 01; here it is just a preview that LM calls compose with everything else.)

Rules of thumb:

- `dspy.configure(...)` — global default, set once per process (notebooks are special-cased so re-running the cell is fine).
- `with dspy.context(...):` — temporary override, correct inside threads/async code too.

**Coming from older DSPy tutorials?** Three things changed in 3.x that make old snippets fail: per-provider clients like `dspy.OpenAI(...)` are gone (use `dspy.LM("provider/model")`); `dspy.Assert`/`dspy.Suggest` are gone (their modern replacements, `dspy.Refine` and `dspy.BestOfN`, appear in notebook 08); and `temperature`/`max_tokens` no longer default to `0.0`/`1000` — unset means "provider default".

## Recap

| Concept | One-liner |
|---|---|
| `dspy.LM("provider/model")` | one wrapper for any model; call it directly with a string |
| `dspy.configure(lm=...)` | set the process-wide default LM |
| `dspy.context(lm=...)` | scoped override for a block |
| cache | on by default, on disk; identical requests are free |
| the big idea | program + data + metric + optimizer, like PyTorch with prompts as weights |

## Exercises

1. Point `MODEL` somewhere else, both ways: (a) uncomment an override line in the config cell and rerun it; (b) set `MODEL=...` in `.env` and restart the kernel — the global route every other notebook will also see. Rerun the raw call each time. Notice the *program* — such as it is — did not change.
2. Call `lm("Name a sea bird.", temperature=1.2)` twice. Why are the two results identical anyway? (Hint: what is the cache key made of?) Now change the prompt by one character and run again.
3. Look at `~/.dspy_cache` on disk. Roughly how would you expect its size to grow over this course?

**Next up:** the first real abstraction — signatures, or how to declare *what* an LM call does instead of *how* to prompt it.